# 01 · Setup de BigQuery — creación del dataset y las tablas

**VoltiaTech** · e-commerce de electrónica y accesorios tecnológicos en Europa.

Este notebook crea la infraestructura: el dataset y las 7 tablas del modelo con sus tipos
de dato, en el orden que respeta las dependencias de claves foráneas.

> **Nota sobre el backend.** El destino de este proyecto es **Google BigQuery**. Para que el
> notebook se pueda ejecutar también sin credenciales de GCP (un compañero que aún no ha
> configurado su service account, una revisión del repo en otra máquina), `src/conexion.py`
> expone la misma interfaz sobre dos implementaciones: BigQuery y un **espejo local de
> SQLite con idéntico esquema**. Si hay credenciales, se usa BigQuery; si no, el espejo.
> Las queries son SQL estándar y corren igual en los dos.

## 1. Imports y configuración

In [1]:
import os
import sys

# el proyecto se importa desde la raiz del repo, no desde notebooks/
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"))   # si no existe, no pasa nada
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

from src.esquema import ESQUEMA, ORDEN_CARGA, campos, ddl_sqlite, foraneas, resumen
from src.conexion import obtener_backend

print("Variables de entorno leídas:")
for clave in ["GCP_PROJECT_ID", "BQ_DATASET_ID", "GOOGLE_APPLICATION_CREDENTIALS"]:
    valor = os.getenv(clave)
    print(f"  {clave:<32} {valor if valor else '(no definida)'}")

Variables de entorno leídas:
  GCP_PROJECT_ID                   (no definida)
  BQ_DATASET_ID                    (no definida)
  GOOGLE_APPLICATION_CREDENTIALS   (no definida)


## 2. El modelo de datos

El esquema vive en un único sitio, `src/esquema.py`, y de ahí se derivan tanto los
`SchemaField` de BigQuery como el DDL de SQLite. Así el diagrama, las tablas reales y el
espejo local no pueden desincronizarse.

In [2]:
print(resumen())

categories      3 campos | PK: category_id    | FKs: -
customers       8 campos | PK: customer_id    | FKs: -
products        7 campos | PK: product_id     | FKs: category_id
orders          8 campos | PK: order_id       | FKs: customer_id
order_items     6 campos | PK: order_item_id  | FKs: order_id, product_id
payments        6 campos | PK: payment_id     | FKs: order_id
reviews         5 campos | PK: review_id      | FKs: order_item_id


### Las 7 entidades y por qué existen

| Tabla | Qué guarda | Por qué es una tabla aparte |
|---|---|---|
| `categories` | Clasificación del catálogo | Evita repetir el nombre de la categoría en cada producto (3NF) |
| `customers` | Datos de contacto y captación | Entidad propia: un cliente existe aunque no haya comprado |
| `products` | Catálogo con precio **y coste** | El coste es imprescindible para calcular margen |
| `orders` | Cabecera del pedido | Un pedido tiene estado y fechas propias, independientes de sus líneas |
| `order_items` | Detalle: qué productos y a qué precio | **Resuelve la relación N:M** entre pedidos y productos |
| `payments` | Cobros y su estado | Un pedido puede tener un pago reembolsado o fallido: es otra realidad |
| `reviews` | Valoraciones | Se valora **una línea concreta**, no el pedido entero |

**La pregunta clave del enunciado:** *¿qué relación hay entre pedidos y productos?*

Es **N:M**: un pedido lleva muchos productos y un producto aparece en muchos pedidos. Una
relación N:M no se puede representar con una FK en ninguno de los dos lados, así que
necesita una **tabla intermedia**: `order_items`. Y esa tabla no es un mero puente, tiene
atributos propios (`quantity`, `unit_price`, `discount_pct`) que solo tienen sentido en la
intersección de un pedido con un producto.

## 3. Conexión al backend

In [3]:
backend = obtener_backend()
print("\nUsando:", backend.nombre)

Backend: SQLite (espejo local). Motivo: GCP_PROJECT_ID no está configurado en el .env.
Las queries son SQL estándar: el resultado es equivalente al de BigQuery.

Usando: SQLite (espejo local)


## 4. Creación del dataset

En BigQuery el dataset se crea con `location="EU"`: son datos de clientes europeos, así que
los mantengo en región europea por coherencia con el RGPD. La llamada es idempotente
(`exists_ok=True`), de modo que volver a ejecutar el notebook no falla.

```python
# lo que hace por dentro BackendBigQuery.crear_dataset()
dataset = bigquery.Dataset(f"{project_id}.{dataset_id}")
dataset.location = "EU"
client.create_dataset(dataset, exists_ok=True)
```

In [4]:
backend.crear_dataset()
print("Dataset listo.")

Dataset listo.


## 5. Creación de las tablas

**El orden importa.** `order_items` referencia a `orders` y a `products`, y `reviews`
referencia a `order_items`. Creando las tablas en orden alfabético, las FK apuntarían a
tablas que todavía no existen. Por eso `ORDEN_CARGA` fija la secuencia correcta:

`categories → customers → products → orders → order_items → payments → reviews`

Así se ve el DDL que se genera para la tabla intermedia:

In [5]:
print(ddl_sqlite("order_items"))

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    discount_pct REAL NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);


In [6]:
creadas = backend.crear_tablas(recrear=True)
print(f"{len(creadas)} tablas creadas, en este orden:")
for i, t in enumerate(creadas, 1):
    fks = foraneas(t)
    detalle = f" (depende de: {', '.join(sorted({f[1] for f in fks}))})" if fks else ""
    print(f"  {i}. {t}{detalle}")

7 tablas creadas, en este orden:
  1. categories
  2. customers
  3. products (depende de: categories)
  4. orders (depende de: customers)
  5. order_items (depende de: orders, products)
  6. payments (depende de: orders)
  7. reviews (depende de: order_items)


## 6. Verificación

Compruebo que las 7 tablas existen, están vacías y tienen las columnas esperadas.

In [7]:
verificacion = []
for tabla in ORDEN_CARGA:
    filas = backend.query(f"SELECT COUNT(*) AS n FROM {backend.tabla(tabla)}")["n"][0]
    verificacion.append({
        "tabla": tabla,
        "columnas_esperadas": len(campos(tabla)),
        "filas": filas,
    })

df_verif = pd.DataFrame(verificacion)
display(df_verif)

assert len(df_verif) == 7, "Deberían existir 7 tablas"
assert df_verif["filas"].sum() == 0, "Las tablas deberían estar vacías tras crearlas"
print("\nInfraestructura verificada: 7 tablas creadas y vacías, listas para la carga.")

,tabla,columnas_esperadas,filas
0,categories,3,0
1,customers,8,0
2,products,7,0
3,orders,8,0
4,order_items,6,0
5,payments,6,0
6,reviews,5,0



Infraestructura verificada: 7 tablas creadas y vacías, listas para la carga.


---

**Siguiente paso:** `02_generate_data.ipynb` genera los datos sintéticos con Faker y los carga.

In [8]:
backend.cerrar()